# R1.5: условная оценка нулевых потерь
Только чтение уже проверенного анализа десяти итоговых реализаций. Новая имитация, bootstrap и обучение не запускаются. Исходный R1.4 сохраняет прежний статус; новая оценка условна по допущению независимости. Откройте блокнот из распакованного комплекта воспроизводимости или рабочей копии ветки.


In [1]:
from pathlib import Path
import hashlib, json, math
roots = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in roots if (p/'results/r15/conditional_analysis.json').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Нужен полный комплект воспроизводимости R1.5, а не только PDF.')
source = ROOT/'results/r15/conditional_analysis.json'
analysis = json.loads(source.read_text(encoding='utf-8'))
raw = json.loads((ROOT/'results/r14/article_evidence.json').read_text(encoding='utf-8'))
assert analysis['source_run_id'] == 45174257711
assert raw['counts']['requests'] == 131 and raw['status'] == 'INSUFFICIENT_DATA'
assert sorted(r['seed'] for r in analysis['rows']) == list(range(10))
assert all(r['arrivals'] == r['completed'] + r['dropped'] + r['unfinished'] for r in analysis['rows'])
assert all(r['dropped'] == r['unfinished'] == 0 for r in analysis['rows'])
print('Источник:', source.relative_to(ROOT))
print('SHA256:', hashlib.sha256(source.read_bytes()).hexdigest())


Источник: results/r15/conditional_analysis.json
SHA256: 11c91a69c81242a5d27fdead0a69e29caa73d2a98b85db250d62be942c33d016


In [2]:
n_i = [r['arrivals'] for r in analysis['rows']]
n_min = min(n_i)
u = -math.expm1(math.log(0.025)/n_min)
assert n_min == analysis['decision']['n_min']
assert abs(u-analysis['decision']['U_L']) < 1e-15
assert sum(n_i) == raw['point']['arrivals']
assert raw['q_interval']['upper'] <= 200 and raw['q_interval']['half_width'] <= 10
assert u <= 0.001 and u/2 <= 0.00005
report = dict(cell_id='r15-05-summary', source_file='results/r15/conditional_analysis.json',
              source_sha256=hashlib.sha256(source.read_bytes()).hexdigest(),
              n_i=n_i, n_min=n_min, U_L=u, U_L_percent=100*u,
              U_Q_ms=raw['q_interval']['upper'],
              conditional_status=analysis['decision']['status'],
              assumption=analysis['decision']['assumption'],
              sensitivity=analysis['sensitivity'], mm1k=analysis['mm1k'],
              raw_R14_status=raw['status'], simulation_executed=False)
print('R15_CONDITIONAL_REPORT='+json.dumps(report,ensure_ascii=False))
print('Вывод: допустима при указанном допущении независимости. Минимальность и независимое подтверждение не установлены.')
print('Уменьшение эффективного объёма в 50 раз сохраняет порог риска, но не отдельный допуск точности.')


R15_CONDITIONAL_REPORT={"cell_id": "r15-05-summary", "source_file": "results/r15/conditional_analysis.json", "source_sha256": "11c91a69c81242a5d27fdead0a69e29caa73d2a98b85db250d62be942c33d016", "n_i": [200298, 199701, 200008, 199114, 200298, 200255, 199726, 199959, 200445, 199997], "n_min": 199114, "U_L": 1.8526297916613364e-05, "U_L_percent": 0.0018526297916613365, "U_Q_ms": 171.01545105250003, "conditional_status": "CONDITIONALLY_ACCEPTABLE", "assumption": "Independent Bernoulli message-loss outcomes with a common probability; dependence of the actual queue has not been established or removed.", "sensitivity": [{"reduction_factor": 1, "n_eff": 199114.0, "U_L": 1.8526297916613364e-05, "loss_threshold_pass": true, "old_precision_pass": true, "interpretation": "assumed effective-volume scenario, not measured dependence"}, {"reduction_factor": 5, "n_eff": 39822.8, "U_L": 9.262805740950791e-05, "loss_threshold_pass": true, "old_precision_pass": true, "interpretation": "assumed effective-v